# Puntos críticos y localidades
Objetivo: encontrar los puntos con más víctimas sin cámara y comparar localidades por cada 100.000 habitantes.
Todo con víctimas de 2019 a 2025.

In [1]:
import pandas as pd
import numpy as np

victimas = pd.read_csv("../data/processed/victimas_camaras.csv.gz")
camaras = pd.read_csv("../data/processed/camaras.csv")
poblacion = pd.read_csv("../data/processed/poblacion.csv")

periodo = victimas[victimas["AA_Acc"] >= 2019].copy()
periodo["muerto"] = periodo["Gravedad_Indicador_30d"] == "MUERTO"
periodo["Localidad"] = periodo["Localidad"].fillna("SIN LOCALIDAD")
print(periodo.shape)

(138178, 18)


# Puntos críticos
Armar los cuadros

In [2]:
# 300 metros a grados: 300 / 111.000 = 0,0027
lado = 0.0027

periodo["fila"] = (periodo["Latitud"] / lado).round().astype(int)
periodo["columna"] = (periodo["Longitud"] / lado).round().astype(int)
periodo["cuadro"] = periodo["fila"].astype(str) + "_" + periodo["columna"].astype(str)

print(periodo["cuadro"].nunique())

3925


In [3]:
cuadros = periodo.groupby("cuadro").agg(
    victimas=("Codigo_Accidentado", "count"),
    muertos=("muerto", "sum"),
    latitud=("Latitud", "mean"),
    longitud=("Longitud", "mean"),
    distancia_min=("distancia_m", "min"),
    localidad=("Localidad", lambda x: x.mode()[0]),
    direccion=("Direccion", lambda x: x.mode()[0]),
    actor_principal=("Condicion", lambda x: x.mode()[0]),
    franja_principal=("franja", lambda x: x.mode()[0])
).reset_index()

cuadros["tiene_camara"] = cuadros["distancia_min"] <= 300

# si dos cuadros tienen las mismas víctimas va primero el de más muertos (y si siguen empatados, por el código del cuadro)
cuadros = cuadros.sort_values(["victimas", "muertos", "cuadro"], ascending=[False, False, True]).reset_index(drop=True)
cuadros["puesto"] = cuadros.index + 1
print(cuadros.shape)

(3925, 12)


Respuesta

In [4]:
criticos = cuadros.head(50)

print("Puntos críticos sin cámara:", (~criticos["tiene_camara"]).sum(), "de 50")
print("Víctimas en los 50 puntos críticos:", criticos["victimas"].sum())

criticos[["puesto", "victimas", "muertos", "direccion", "localidad", "actor_principal", "tiene_camara"]].head(10)

Puntos críticos sin cámara: 34 de 50
Víctimas en los 50 puntos críticos: 10581


,puesto,victimas,muertos,direccion,localidad,actor_principal,tiene_camara
0,1,260,7,AV AVENIDA DE LAS AMERICAS - KR 53 02,PUENTE ARANDA,MOTOCICLISTA,False
1,2,257,5,AU SUR - KR 76 02,BOSA,MOTOCICLISTA,True
2,3,255,10,AV AVENIDA BOYACA - CL 44 S 02,KENNEDY,MOTOCICLISTA,False
3,4,254,18,AVENIDA BOYACA 4 SUR 02,KENNEDY,MOTOCICLISTA,False
4,5,252,10,AU SUR - AV AVENIDA PRIMERO DE MAYO 02,ANTONIO NARIÑO,MOTOCICLISTA,False
5,6,252,5,AV AVENIDA DE LAS AMERICAS - KR 50 02,PUENTE ARANDA,MOTOCICLISTA,False
6,7,248,10,KR 86 - CL 26 02,FONTIBÓN,MOTOCICLISTA,False
7,8,247,5,KR 14 - CL 22 02,SANTA FE,MOTOCICLISTA,False
8,9,241,5,AV AVENIDA CIUDAD DE CALI - AV AVENIDA CIUDAD ...,KENNEDY,MOTOCICLISTA,True
9,10,240,5,CL 145 - KR 104 02,SUBA,MOTOCICLISTA,True


In [5]:
# ¿a cuántos puntos críticos llegan las cámaras nuevas de 2026?
# una víctima está cerca de una cámara oficial si está a 300 m o menos y su cámara más cercana no es de 2026
periodo["cerca_oficial"] = (periodo["distancia_m"] <= 300) & (~periodo["id_camara"].str.startswith("NUEVA"))
con_oficial = periodo.groupby("cuadro").agg(con_oficial=("cerca_oficial", "max")).reset_index()
criticos2 = criticos.merge(con_oficial, on="cuadro", how="left", validate="1:1")
print("Sin cámara contando solo la capa oficial:", (~criticos2["con_oficial"]).sum(), "de 50")
print("Sin cámara contando también las de 2026:", (~criticos2["tiene_camara"]).sum(), "de 50")

Sin cámara contando solo la capa oficial: 38 de 50
Sin cámara contando también las de 2026: 34 de 50


Quienes y cuándo en los puntos sin cámara

In [6]:
sin_camara = criticos[~criticos["tiene_camara"]]

print(sin_camara["actor_principal"].value_counts())
print(sin_camara["franja_principal"].value_counts())
print(sin_camara["localidad"].value_counts())

actor_principal
MOTOCICLISTA    34
Name: count, dtype: int64
franja_principal
Mañana (6-11)      16
Noche (18-23)       9
Tarde (12-17)       7
Madrugada (0-5)     2
Name: count, dtype: int64
localidad
PUENTE ARANDA         5
KENNEDY               4
FONTIBÓN              4
TUNJUELITO            4
BOSA                  4
ANTONIO NARIÑO        3
SANTA FE              3
SAN CRISTÓBAL         2
CIUDAD BOLÍVAR        2
LOS MÁRTIRES          1
RAFAEL URIBE URIBE    1
ENGATIVÁ              1
Name: count, dtype: int64


¿las cámaras nuevas de 2026 quedaron en puntos críticos?

In [7]:
camaras["fila"] = (camaras["latitud"] / lado).round().astype(int)
camaras["columna"] = (camaras["longitud"] / lado).round().astype(int)
camaras["cuadro"] = camaras["fila"].astype(str) + "_" + camaras["columna"].astype(str)

nuevas = camaras[camaras["id_camara"].str.startswith("NUEVA")]
nuevas = nuevas.merge(cuadros[["cuadro", "puesto", "victimas"]], on="cuadro", how="left")
nuevas[["id_camara", "direccion", "puesto", "victimas"]]

,id_camara,direccion,puesto,victimas
0,NUEVA-01,AV NQS - CL 25A (N-S),51,172
1,NUEVA-02,AV NQS - CL 9 (N-S),83,154
2,NUEVA-03,AV NQS - CL 9 (S-N),83,154
3,NUEVA-04,AV NQS - CL 47A (N-S),467,77
4,NUEVA-05,AV NQS - CL 47A (S-N),467,77
5,NUEVA-06,AV AMERICAS - KR 58 (O-E),78,155
6,NUEVA-07,AV AMERICAS - KR 58 (E-O),78,155
7,NUEVA-08,AV EL DORADO - KR 17 (O-E),723,59
8,NUEVA-09,AV EL DORADO - KR 25 (E-O),373,87


Ningún cuadro con cámara nueva está entre los 50 primeros. El mejor es la NQS con calle 25A, en el puesto 51.

Guardar

In [8]:
cuadros.to_csv("../data/processed/cuadros.csv", index=False)

Víctimas por localidad y año, unidas con la población

In [9]:
por_anio = periodo[periodo["Localidad"] != "SIN LOCALIDAD"].groupby(["Localidad", "AA_Acc"]).agg(
    victimas=("Codigo_Accidentado", "count"),
    muertos=("muerto", "sum")
).reset_index()
por_anio = por_anio.rename(columns={"Localidad": "localidad", "AA_Acc": "anio"})
print(por_anio.shape)

por_anio = por_anio.merge(poblacion, on=["localidad", "anio"], how="left", validate="1:1", indicator=True)
print(por_anio["_merge"].value_counts())
por_anio = por_anio.drop(columns="_merge")

por_anio["tasa_victimas"] = por_anio["victimas"] / por_anio["poblacion"] * 100000
por_anio.to_csv("../data/processed/localidades_anio.csv", index=False)

(139, 4)
_merge
both          139
left_only       0
right_only      0
Name: count, dtype: int64


Resumen por localidad con cámaras

In [10]:
localidades = por_anio.groupby("localidad").agg(
    victimas=("victimas", "sum"),
    muertos=("muertos", "sum"),
    poblacion_suma=("poblacion", "sum")
).reset_index()

# victimas por cada 100.000 habitantes al año (promedio 2019-2025)
localidades["tasa_victimas"] = localidades["victimas"] / localidades["poblacion_suma"] * 100000
localidades["tasa_muertos"] = localidades["muertos"] / localidades["poblacion_suma"] * 100000

# cuantas camaras hay en cada localidad
cuantas = camaras.groupby("localidad").agg(camaras=("id_camara", "count")).reset_index()
localidades = localidades.merge(cuantas, on="localidad", how="left", validate="1:1")
localidades["camaras"] = localidades["camaras"].fillna(0)

# victimas por cámara (si no hay cámaras, se deja vacío)
localidades["victimas_por_camara"] = np.where(localidades["camaras"] > 0,
                                              localidades["victimas"] / localidades["camaras"], np.nan)

localidades = localidades.sort_values("tasa_victimas", ascending=False)
localidades.to_csv("../data/processed/localidades.csv", index=False)
localidades.round(1)

,localidad,victimas,muertos,poblacion_suma,tasa_victimas,tasa_muertos,camaras,victimas_por_camara
9,LOS MÁRTIRES,4756,120,581791,817.5,20.6,0.0,NaN
0,ANTONIO NARIÑO,3708,88,581749,637.4,15.1,1.0,3708.0
13,SANTA FE,4750,140,752232,631.5,18.6,2.0,2375.0
3,CANDELARIA,738,14,127001,581.1,11.0,0.0,NaN
10,PUENTE ARANDA,10194,240,1780774,572.4,13.5,8.0,1274.2
16,TEUSAQUILLO,5727,131,1145422,500.0,11.4,9.0,636.3
1,BARRIOS UNIDOS,4926,90,1047381,470.3,8.6,4.0,1231.5
4,CHAPINERO,4736,77,1229024,385.3,6.3,5.0,947.2
17,TUNJUELITO,4886,127,1269344,384.9,10.0,2.0,2443.0
7,FONTIBÓN,8482,258,2780579,305.0,9.3,2.0,4241.0


## Hallazgos (2019-2025)
- Dividimos la ciudad en cuadros de 300 m: 3.925 cuadros tuvieron al menos una víctima.
- De los 50 puntos con más víctimas, 34 no tienen una cámara a 300 m o menos.
- En los 34 puntos sin cámara la víctima más común es el motociclista (50,2 % de las víctimas en esos puntos).
- El punto con más víctimas es la Av. de Las Américas con carrera 53 con 260 víctimas y no tiene cámara.
- Las cámaras nuevas de 2026 quedan a 300 m o menos de 4 puntos críticos: sin ellas habría 38 sin cámara y con ellas quedan 34.
- Usme no tiene cámaras y tuvo 4.500 víctimas. Fontibón tiene 2 cámaras y 8.482 víctimas.

- - - - -
- Las tasas por localidad usan la población que vive ahí, no la que circula. Por eso las localidades del centro salen altas.